# ClearRoute — Continuar Treino com Imagens Próprias
Este notebook continua o treino a partir do `best.pt` (já treinado no Trash-AI)
usando novas imagens anotadas no Roboflow.

Corre as células **de cima para baixo**, por ordem.

## Célula 1 — Verificar GPU
Antes de qualquer coisa, confirma que o Colab te deu uma GPU.
Se aparecer **AVISO**, vai a **Runtime → Change runtime type → T4 GPU → Save** e corre outra vez.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU disponível: {torch.cuda.get_device_name(0)}")
else:
    print("AVISO: Sem GPU! Vai em Runtime → Change runtime type → T4 GPU")

## Célula 2 — Instalar Dependências
Instala o `ultralytics` (YOLO11) e o `roboflow` (download do dataset).
O `!` no início da linha significa: corre este comando no terminal, não em Python.

In [ ]:
!pip install ultralytics roboflow -q
from ultralytics import YOLO
import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")

## Célula 3 — Upload do best.pt
Faz upload do modelo que já treinaste anteriormente (`best.pt`).
Este modelo já sabe reconhecer lixo — vamos continuar o treino com as tuas próprias imagens.

In [ ]:
from google.colab import files
print("Faz upload do teu best.pt agora:")
uploaded = files.upload()

# Confirma que foi recebido
import os
if os.path.exists("best.pt"):
    print("best.pt recebido com sucesso!")
else:
    print("ERRO: best.pt não encontrado. Tenta de novo.")

## Célula 4 — Baixar Dataset do Roboflow
Faz o download do dataset com as tuas imagens anotadas.

**Antes de correr esta célula:**
1. Vai a [roboflow.com](https://roboflow.com) → Settings → API Keys
2. Copia a tua **Private API Key**
3. Cola-a abaixo a substituir `COLE_TUA_API_KEY_AQUI`

In [ ]:
!pip install roboflow -q
from roboflow import Roboflow

# SUBSTITUI esta linha com a tua API key do Roboflow
rf = Roboflow(api_key="COLE_TUA_API_KEY_AQUI")

project = rf.workspace("arthur-taha").project("clearroute_extra")
version = project.version(1)
dataset = version.download("yolov11")
print(f"Dataset baixado em: {dataset.location}")
print(f"Config file: {dataset.location}/data.yaml")

## Célula 5 — Verificar o Dataset
Confirma que as imagens e anotações chegaram correctamente.
Deves ver o número de imagens de treino e validação, e os nomes das classes.

In [ ]:
import yaml
with open(f"{dataset.location}/data.yaml") as f:
    data = yaml.safe_load(f)
print(f"Classes ({data['nc']}): {data['names']}")

import os
train_imgs = len(os.listdir(f"{dataset.location}/train/images"))
valid_imgs = len(os.listdir(f"{dataset.location}/valid/images"))
print(f"Treino: {train_imgs} imagens")
print(f"Validação: {valid_imgs} imagens")

## Célula 6 — Continuar Treino a partir do best.pt
A diferença chave: em vez de `YOLO('yolo11n.pt')` (modelo genérico do zero),
usamos `YOLO('best.pt')` — o modelo já treinado no Trash-AI.

Isto chama-se **fine-tuning continuado**: o modelo já sabe o que é lixo
e agora aprende também com as tuas imagens locais.

| Parâmetro | Valor | Porquê |
|-----------|-------|--------|
| `epochs` | 40 | Mais do que o 1.º treino — dataset mais pequeno precisa de mais épocas |
| `patience` | 15 | Para automaticamente se não melhorar durante 15 épocas |
| `mosaic` | 1.0 | Junta 4 imagens numa — compensa datasets pequenos |
| `mixup` | 0.1 | Mistura duas imagens — reduz overfitting |

In [ ]:
# IMPORTANTE: começamos do best.pt, não do yolo11n.pt genérico
# Isso preserva tudo que o modelo já aprendeu no Trash-AI
model = YOLO("best.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=40,
    imgsz=640,
    batch=16,
    patience=15,
    project="clearroute_v2",
    name="resume_local_images",
    exist_ok=True,
    # Augmentações extras pra compensar o dataset pequeno
    degrees=10.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)
print("Treino concluído!")

## Célula 7 — Comparar Métricas: best.pt vs best2.pt
Avalia os dois modelos nas mesmas imagens de validação e mostra a diferença.

- **mAP50** — métrica principal (0 a 1). Acima de 0.5 é bom; acima de 0.7 é excelente.
- **mAP50-95** — métrica mais rigorosa (média entre vários limiares de sobreposição).

In [ ]:
# Avalia o modelo NOVO nas imagens de validação
new_model = YOLO("/content/clearroute_v2/resume_local_images/weights/best.pt")
new_metrics = new_model.val(data=f"{dataset.location}/data.yaml", verbose=False)
print(f"Novo modelo — mAP50: {new_metrics.box.map50:.3f}")
print(f"Novo modelo — mAP50-95: {new_metrics.box.map:.3f}")

# Avalia o modelo ORIGINAL para comparação
old_model = YOLO("best.pt")
old_metrics = old_model.val(data=f"{dataset.location}/data.yaml", verbose=False)
print(f"Modelo original — mAP50: {old_metrics.box.map50:.3f}")
print(f"Modelo original — mAP50-95: {old_metrics.box.map:.3f}")

diff = new_metrics.box.map50 - old_metrics.box.map50
print(f"Diferença: {diff:+.3f} ({'melhorou' if diff > 0 else 'piorou'})")

## Célula 8 — Baixar o best2.pt
Copia o melhor modelo desta sessão para `best2.pt` e faz download.
Guarda-o na pasta `clearroute/models/` do teu projecto.

In [ ]:
import shutil
shutil.copy(
    "/content/clearroute_v2/resume_local_images/weights/best.pt",
    "/content/best2.pt"
)
files.download("/content/best2.pt")
print("Download do best2.pt iniciado!")
print("No detect.py, troca: model = YOLO('models/best.pt') → model = YOLO('models/best2.pt')")

---
## O que fazer a seguir

### O que é o best2.pt?
`best2.pt` é uma versão melhorada do modelo original (`best.pt`).
Enquanto `best.pt` foi treinado apenas no dataset público Trash-AI,
`best2.pt` foi continuado com as tuas próprias imagens anotadas —
o que o torna mais preciso nas condições reais do ClearRoute
(ângulos de câmara, iluminação, tipos de rua em Konstanz).

### Como usar o best2.pt no detect.py
Só precisas de mudar **uma linha** em `detect.py`:
```python
# Antes
model = YOLO("models/best.pt")

# Depois
model = YOLO("models/best2.pt")
```
Move o ficheiro `best2.pt` para a pasta `clearroute/models/` e o resto funciona igual.

### O que fazer quando chegarem mais imagens
Se anotares mais imagens no Roboflow e criares uma nova versão do dataset:
1. Vai à **Célula 4** e muda `project.version(1)` para o número da nova versão
2. Corre de novo a partir da Célula 4
3. Na **Célula 3**, faz upload do `best2.pt` mais recente para continuar a partir dele

Cada ciclo de **anotação → treino → avaliação** melhora o modelo progressivamente.